In [ ]:
from pathlib import Path
import sqlalchemy as sa

from omop_semantics.utils.load import load as load_semantics
from omop_constructs.schema.manifest_load import load_manifest
from omop_constructs.factories.from_manifest import FactoryContext, build_from_manifest
from omop_constructs.core.registry import ConstructRegistry

SCHEMA = [Path(".../omop_semantic_core.yaml"), Path(".../omop_staging.yaml")]
INST   = [Path(".../unknowns.yaml"), Path(".../staging_axis.yaml")]
MAN    = Path(".../constructs_manifest.yaml")

sem = load_semantics(schema_paths=SCHEMA, instance_paths=INST)
manifest = load_manifest(MAN)

engine = sa.create_engine("postgresql+psycopg://...")

ctx = FactoryContext(
    semantics=sem,
    episode_of_care_concept_id=32533,  # DiseaseEpisodeConcepts.episode_of_care
    tables={
      # in practice, populate from your constructs bundle:
      "ModifiedCondition": ModifiedCondition,
      "SurgicalProcedure": SurgicalProcedure,
      "SACTRegimen": SACTRegimen,
      "RTCourse": RTCourse,
    },
)

constructs = build_from_manifest(manifest=manifest, ctx=ctx)
registry = ConstructRegistry(constructs)

print(registry.plan())

with engine.begin() as conn:
    registry.create_all(conn, with_data=True)
    registry.refresh_all(conn)
